In [15]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Cell 2 — Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

GPU available: True
Device: Tesla T4


In [17]:
# Cell 3 — Install dependencies
!pip install -q transformers==4.38.2

In [18]:
# Cell 4 — Load WavLM-Large (enhanced from wavlm-base)
# Enhancement 1: WavLM-Large gives [T, 1024] instead of [T, 768]
# Enhancement 2: Weighted layer combination uses all 25 layers

from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import torch
import torch.nn as nn

device = 'cuda'

# ← Changed from wavlm-base to wavlm-large
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-large')
model             = WavLMModel.from_pretrained('microsoft/wavlm-large').to(device)
model.eval()

# WavLM-Large has 24 transformer layers + 1 input CNN layer = 25 total
# Learnable weights — one per layer
layer_weights = nn.Parameter(torch.ones(25, device=device))

print('✅ WavLM-Large loaded!')
print('   Hidden size: 1024 (was 768 in Base)')
print('   Total layers: 25')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at microsoft/wavlm-large were not used when initializing WavLMModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing WavLMModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing WavLMModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of WavLMModel were not initialized from the model checkpoint a

✅ WavLM-Large loaded!
   Hidden size: 1024 (was 768 in Base)
   Total layers: 25


In [19]:
# Cell 5 — Define paths
import os
import numpy as np
import pandas as pd

DRIVE_ROOT           = '/content/drive/MyDrive/Hackenza_MUPlovers'
DATA_PATH            = os.path.join(DRIVE_ROOT, 'data')
PROCESSED_AUDIO_PATH = os.path.join(DATA_PATH, 'processed')
CHUNK_INDEX_PATH     = os.path.join(DATA_PATH, 'chunk_index.csv')

# ← New folder for enhanced embeddings — keeps old embeddings safe
EMBED_SAVE_PATH = os.path.join(DRIVE_ROOT, 'cache', 'embeddings_new')
os.makedirs(EMBED_SAVE_PATH, exist_ok=True)

chunk_df = pd.read_csv(CHUNK_INDEX_PATH)

print('Drive root           :', DRIVE_ROOT)
print('Processed audio path :', PROCESSED_AUDIO_PATH)
print('Chunk index path     :', CHUNK_INDEX_PATH)
print('Embedding save path  :', EMBED_SAVE_PATH)
print('Total chunks         :', len(chunk_df))
print('Unique files         :', chunk_df['file_id'].nunique())

Drive root           : /content/drive/MyDrive/Hackenza_MUPlovers
Processed audio path : /content/drive/MyDrive/Hackenza_MUPlovers/data/processed
Chunk index path     : /content/drive/MyDrive/Hackenza_MUPlovers/data/chunk_index.csv
Embedding save path  : /content/drive/MyDrive/Hackenza_MUPlovers/cache/embeddings_new
Total chunks         : 5179
Unique files         : 160


In [21]:
# Cell 6 — Define enhanced chunk embedding extraction
# Enhancement 1: Uses WavLM-Large (1024-dim)
# Enhancement 2: Weighted combination of ALL 25 layers instead of just last layer

def extract_chunk_embedding(chunk_audio, feature_extractor, model, device='cuda'):
    """
    Extracts embedding from a single audio chunk.
    Uses weighted combination of all 25 WavLM-Large layers.
    Returns tensor of shape [1024]
    """
    inputs = feature_extractor(
        chunk_audio.cpu().numpy(),
        sampling_rate=16000,
        return_tensors='pt'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        # output_hidden_states=True gets ALL 25 layers
        outputs = model(**inputs, output_hidden_states=True)

    # Stack all 25 hidden states → [25, 1, T, 1024]
    all_hidden = torch.stack(outputs.hidden_states)  # [25, 1, T_frames, 1024]

    # Weighted combination across layers
    # softmax ensures weights sum to 1
    weights    = torch.softmax(layer_weights, dim=0)  # [25]
    weights    = weights.view(25, 1, 1, 1)            # reshape for broadcasting

    # Weighted sum across all 25 layers
    embedding  = (all_hidden * weights).sum(dim=0)    # [1, T_frames, 1024]

    # Mean pool across time frames
    embedding  = embedding.mean(dim=1)                # [1, 1024]
    embedding  = embedding.squeeze(0)                 # [1024]

    return embedding.detach().cpu()

print('✅ Enhanced embedding function defined!')
print('   Output dim: 1024 per chunk')

✅ Enhanced embedding function defined!
   Output dim: 1024 per chunk


In [22]:
# Cell 7 — Define file-level embedding extraction
import torchaudio

def extract_embeddings_for_file(file_id):
    """
    Extracts embeddings for all chunks of one file.
    Returns tensor of shape [T, 1024]
    """
    audio_path = os.path.join(PROCESSED_AUDIO_PATH, f'{file_id}.wav')
    waveform, sr = torchaudio.load(audio_path)

    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0)

    file_chunks = chunk_df[chunk_df['file_id'] == int(file_id)]
    embeddings  = []

    for _, row in file_chunks.iterrows():
        start_sample = int(row['start_sample'])
        end_sample   = int(row['end_sample'])
        chunk_audio  = waveform[start_sample:end_sample]

        # Ensure exactly 3 seconds = 48000 samples
        expected_len = 48000
        if len(chunk_audio) < expected_len:
            padding     = expected_len - len(chunk_audio)
            chunk_audio = torch.nn.functional.pad(chunk_audio, (0, padding))

        embedding = extract_chunk_embedding(
            chunk_audio,
            feature_extractor,
            model,
            device=device
        )
        embeddings.append(embedding)

    embeddings = torch.stack(embeddings)  # [T, 1024]

    assert embeddings.shape[0] == len(file_chunks), \
        f'Chunk count mismatch for {file_id}!'
    assert embeddings.shape[1] == 1024, \
        f'Wrong embedding dim for {file_id}: {embeddings.shape[1]}'

    return embeddings

print('✅ File embedding function defined!')

✅ File embedding function defined!


In [23]:
# Cell 8 — Test on one file first
test_file = '288'

print(f'Testing on file {test_file}...')
emb = extract_embeddings_for_file(test_file)

print(f'✅ Embedding shape: {emb.shape}')
print(f'   Expected       : [T, 1024]')
print(f'   T (chunks)     : {emb.shape[0]}')
print(f'   Dim            : {emb.shape[1]}')

assert emb.shape[1] == 1024, '❌ Wrong dim! Should be 1024'
print('\n✅ Test passed! Proceeding to full extraction...')

Testing on file 288...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


✅ Embedding shape: torch.Size([24, 1024])
   Expected       : [T, 1024]
   T (chunks)     : 24
   Dim            : 1024

✅ Test passed! Proceeding to full extraction...


In [24]:
# Cell 9 — Extract embeddings for ALL 160 files
# Saves to cache/embeddings_new/ — does NOT overwrite old embeddings
from tqdm import tqdm

all_files = sorted([
    f.replace('.wav', '')
    for f in os.listdir(PROCESSED_AUDIO_PATH)
    if f.endswith('.wav')
])

print(f'Total files to process: {len(all_files)}')
print(f'Saving to: {EMBED_SAVE_PATH}')
print(f'Each file will be [T, 1024]\n')

failed = []

for file_id in tqdm(all_files):
    save_path = os.path.join(EMBED_SAVE_PATH, f'{file_id}.npy')

    # Skip if already done
    if os.path.exists(save_path):
        continue

    try:
        emb = extract_embeddings_for_file(file_id)
        np.save(save_path, emb.cpu().numpy())
    except Exception as e:
        print(f'❌ Failed {file_id}: {e}')
        failed.append(file_id)

print(f'\n✅ Done! {len(all_files)-len(failed)}/{len(all_files)} files saved')
if failed:
    print('❌ Failed files:', failed)

Total files to process: 160
Saving to: /content/drive/MyDrive/Hackenza_MUPlovers/cache/embeddings_new
Each file will be [T, 1024]



100%|██████████| 160/160 [05:30<00:00,  2.07s/it]


✅ Done! 160/160 files saved


In [25]:
# Cell 10 — Verify ALL files are same dimension [T, 1024]
print('Verifying all files...')

all_saved = os.listdir(EMBED_SAVE_PATH)
issues    = []
shapes    = []

for fname in tqdm(all_saved):
    if not fname.endswith('.npy'):
        continue
    emb = np.load(os.path.join(EMBED_SAVE_PATH, fname))
    if emb.shape[1] != 1024:
        issues.append(f'{fname} → {emb.shape}')
    else:
        shapes.append(emb.shape)

print(f'\n=== Verification Results ===')
print(f'Total files     : {len(shapes)}')
print(f'All dim = 1024  : {len(issues) == 0}')
if shapes:
    T_vals = [s[0] for s in shapes]
    print(f'T range         : min={min(T_vals)}, max={max(T_vals)}, avg={sum(T_vals)//len(T_vals)}')

if issues:
    print(f'\n❌ Issues found:')
    for i in issues:
        print(f'   {i}')
else:
    print(f'\n✅ All files are [T, 1024] — consistent dimensions!')
    print(f'\n→ Tell Aalu to run Feature Assembly with:')
    print(f'   EMB_DIR = cache/embeddings_new/')
    print(f'   New feature dim = 1024 + 10 + 5 = 1039')

Verifying all files...


100%|██████████| 160/160 [00:00<00:00, 178.08it/s]


=== Verification Results ===
Total files     : 160
All dim = 1024  : True
T range         : min=12, max=267, avg=32

✅ All files are [T, 1024] — consistent dimensions!

→ Tell Aalu to run Feature Assembly with:
   EMB_DIR = cache/embeddings_new/
   New feature dim = 1024 + 10 + 5 = 1039
